# WP-C1 decisive grid, shard 36/40

Frozen under preregistration.md (2026-08-24). Contains
c1 grid cells plus folded C2(i) null cells; checkpoints
every 25 reps; do not edit parameters.

Cells (8):
- null_c4.0 (c2i)
- c4.0_m0.20_partial_r3 (c1)
- c4.0_m1.20_orthogonal_r1 (c1)
- c4.0_m2.40_partial_r3 (c1)
- c0.5_m0.80_partial_r3 (c1)
- c0.5_m2.00_partial_r1 (c1)
- c2.0_m0.60_orthogonal_r3 (c1)
- c2.0_m1.80_orthogonal_r1 (c1)

In [ ]:
# Auto-generated single-file module (source of truth:
# code/scm_frontier/* via scripts/build_colab_notebooks.py)
%%writefile scm_frontier_flat.py
"""scm_frontier_flat: single-file Phase C module (auto-generated; DO NOT EDIT)."""

# ===== flattened from code/scm_frontier/dgps.py =====
"""Factor-panel data-generating processes for the Idea 5 project.

Model (model_card.md): Y_it = L_it + E_it, L_it = sum_k A_ik f_kt with
standardized factor scores, spike strengths s_j = ||a^(j)||^2 / sigma^2,
aspect ratio c = n / T0. Unit 0 is treated; units 1..n-1 are donors.
All spectral diagnostics must use pre-period columns only.
"""

from __future__ import annotations

import numpy as np

NOISE_LAWS = ("gaussian", "ar1", "heteroskedastic")
ALIGNMENTS = ("none", "first", "all")


def _draw_noise(rng, law, shape, sigma, rho, het_ratio):
    if law == "gaussian":
        return rng.normal(0.0, sigma, size=shape)
    if law == "ar1":
        e = np.empty(shape)
        innov_sd = sigma * np.sqrt(1.0 - rho**2)
        e[:, 0] = rng.normal(0.0, sigma, size=shape[0])
        for t in range(1, shape[1]):
            e[:, t] = rho * e[:, t - 1] + rng.normal(0.0, innov_sd, size=shape[0])
        return e
    if law == "heteroskedastic":
        base = rng.normal(0.0, sigma, size=shape)
        hi = rng.random(shape[0]) < 0.5
        scale = np.where(hi, np.sqrt(het_ratio), 1.0 / np.sqrt(het_ratio))
        return base * scale[:, None]
    raise ValueError(f"unknown noise law {law!r}; expected one of {NOISE_LAWS}")


def generate_panel(
    n: int = 121,
    T0: int = 240,
    T_post: int = 100,
    r: int = 1,
    spike_strengths=(6.0,),
    treated_share=(0.5,),
    alignment: str = "first",
    sigma: float = 1.0,
    noise: str = "gaussian",
    rho: float = 0.5,
    het_ratio: float = 4.0,
    structural_break: float | None = None,
    seed: int = 0,
) -> dict:
    """Generate one synthetic panel.

    Parameters
    ----------
    n : total units including the treated unit at row 0.
    T0, T_post : pre- and post-period counts; c = n / T0.
    r : number of factors.
    spike_strengths : donor-carried s_j = ||a^(j)||^2 / sigma^2 per factor.
    treated_share : theta_j = alpha_j^2 / sigma^2 per factor; combined with
        `alignment`: "none" zeroes all alpha, "first" puts leverage only on
        factor 0, "all" spreads it equally across factors.
    structural_break : if not None, post-period factor scores are shifted by
        this amount per unit variance (violates assumption A4).
    seed : numpy default_rng seed; identical seeds give bitwise-identical panels.

    Returns
    -------
    dict with keys Y (n x (T0+T_post)), L, E, F ((T0+T_post) x r), A (n x r),
    and the config echo.
    """
    if noise not in NOISE_LAWS:
        raise ValueError(f"unknown noise law {noise!r}")
    if alignment not in ALIGNMENTS:
        raise ValueError(f"unknown alignment {alignment!r}; expected {ALIGNMENTS}")
    if len(spike_strengths) != r or len(treated_share) != r:
        raise ValueError("spike_strengths and treated_share must have length r")
    rng = np.random.default_rng(seed)
    n_d = n - 1
    T = T0 + T_post

    A = np.zeros((n, r))
    for j, sj in enumerate(spike_strengths):
        A[1:, j] = rng.normal(0.0, np.sqrt(sj * sigma**2 / n_d), size=n_d)
    if alignment == "none":
        pass
    elif alignment == "first":
        A[0, :] = 0.0
        A[0, 0] = sigma * np.sqrt(treated_share[0])
    else:
        A[0, :] = [sigma * np.sqrt(th / r) for th in treated_share]

    F = rng.normal(0.0, 1.0, size=(T, r))
    if structural_break is not None:
        F[T0:, :] += structural_break

    L = A @ F.T
    E = _draw_noise(rng, noise, (n, T), sigma, rho, het_ratio)
    Y = L + E
    config = dict(
        n=n, T0=T0, T_post=T_post, c=n / T0, r=r,
        spike_strengths=tuple(spike_strengths), treated_share=tuple(treated_share),
        alignment=alignment, sigma=sigma, noise=noise, rho=rho,
        het_ratio=het_ratio, structural_break=structural_break, seed=seed,
    )
    return {"Y": Y, "L": L, "E": E, "F": F, "A": A, "config": config}


# ===== flattened from code/scm_frontier/estimators.py =====
"""Estimator library for the Idea 5 project (WP-B2).

Leakage rule (G0): estimators receive ONLY the donor matrix over the full
timeline and the treated pre-period row. The treated post-period outcomes are
never passed in, so leakage is structurally impossible; the test suite
additionally verifies behavioral invariance to treated-post perturbations.

Estimand: realized outcome y*_t = L_0,t + E_0,t for t in the post window.
The oracle therefore predicts L_0,post exactly and attains RMSE = sigma.

All estimators share the signature f(donors, y1_pre, *, info=None, **params)
and return a length-T_post vector of predictions. `info`, when given, is an
optional dict filled with method-specific diagnostics.
"""

from __future__ import annotations

import numpy as np
from scipy.optimize import minimize

DEFAULT_RIDGE_LAMBDAS = tuple(np.logspace(-1.0, 4.0, 11))
DEFAULT_MC_LAMBDAS = tuple(np.logspace(0.0, 3.0, 11))


def donor_mean(donors: np.ndarray, y1_pre: np.ndarray, info: dict | None = None) -> np.ndarray:
    """Uniform-weight donor average, one number per post period."""
    return donors[:, y1_pre.shape[0] :].mean(axis=0)


def oracle_predict(L_post_row: np.ndarray) -> np.ndarray:
    """Oracle uses the true latent trajectory of the treated unit."""
    return L_post_row


def scm_simplex(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    maxiter: int = 500,
    ftol: float = 1e-12,
    info: dict | None = None,
) -> np.ndarray:
    """Abadie SC: least squares under w >= 0, sum(w) = 1 via SLSQP."""
    T0 = y1_pre.shape[0]
    X = donors[:, :T0].T
    n_d = X.shape[1]

    def obj(w):
        r = X @ w - y1_pre
        return float(r @ r)

    def jac(w):
        return 2.0 * X.T @ (X @ w - y1_pre)

    w0 = np.full(n_d, 1.0 / n_d)
    res = minimize(
        obj, w0, jac=jac, method="SLSQP",
        bounds=[(0.0, 1.0)] * n_d,
        constraints=[
            {"type": "eq", "fun": lambda w: w.sum() - 1.0, "jac": lambda w: np.ones(n_d)}
        ],
        options={"maxiter": maxiter, "ftol": ftol},
    )
    if info is not None:
        feasible = abs(res.x.sum() - 1.0) < 1e-6 and (res.x > -1e-8).all()
        info["scm_success"] = bool(res.success or (res.status == 8 and feasible))
        info["scm_status"] = int(res.status)
    return res.x @ donors[:, T0:]


def _cv_ridge_lambda(X, y, lambdas, folds, seed):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    perm = rng.permutation(n)
    blocks = np.array_split(perm, folds)
    eye = np.eye(X.shape[1])
    best_lam, best_err = lambdas[0], np.inf
    for lam in lambdas:
        err = 0.0
        for b in blocks:
            tr = np.setdiff1d(perm, b)
            mux = X[tr].mean(axis=0)
            muy = y[tr].mean()
            Xt = X[tr] - mux
            yt = y[tr] - muy
            coef = np.linalg.solve(Xt.T @ Xt + lam * eye, Xt.T @ yt)
            pred = (X[b] - mux) @ coef + muy
            err += float(np.sum((pred - y[b]) ** 2))
        if err < best_err:
            best_err, best_lam = err, lam
    return best_lam


def ridge_sc(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    lambdas=DEFAULT_RIDGE_LAMBDAS,
    folds: int = 4,
    cv_seed: int = 1234,
    info: dict | None = None,
) -> np.ndarray:
    """Ridge SC: treated row on donor rows with intercept; penalty by CV over
    pre periods only."""
    T0 = y1_pre.shape[0]
    X = donors[:, :T0].T
    lam = _cv_ridge_lambda(X, y1_pre, list(lambdas), folds, cv_seed)
    mux, muy = X.mean(axis=0), y1_pre.mean()
    Xt, yt = X - mux, y1_pre - muy
    coef = np.linalg.solve(Xt.T @ Xt + lam * np.eye(X.shape[1]), Xt.T @ yt)
    if info is not None:
        info["ridge_lambda"] = float(lam)
    return (donors[:, T0:].T - mux) @ coef + muy


def unit_scatter(Y_pre: np.ndarray) -> np.ndarray:
    """(1/T0) * Y Y' in unit space; model-card calibration convention."""
    T0 = Y_pre.shape[1]
    return (Y_pre @ Y_pre.T) / T0


def select_rank_gap_ratio(eigs_desc: np.ndarray, k_max: int) -> int:
    """Rank by largest successive eigenvalue-gap ratio, k in 1..k_max."""
    km = min(k_max, len(eigs_desc) - 1)
    ratios = eigs_desc[:km] / eigs_desc[1 : km + 1]
    return int(np.argmax(ratios)) + 1


def rank_selector(
    donors_pre: np.ndarray, sigma: float, c: float, k_max: int = 4
) -> int:
    """Gap-ratio selector with a TW-style silence gate at the MP edge.

    Returns 0 while the top normalized eigenvalue sits inside the noise band
    (lambda_hat <= 1.05 * sigma^2 (1+sqrt(c))^2, witness-P5 rule); otherwise
    returns the largest-gap position up to k_max. Proper TW calibration is a
    Phase C deliverable (WP-C3).
    """
    evals = np.sort(np.linalg.eigvalsh(unit_scatter(donors_pre)))[::-1]
    edge = sigma**2 * (1.0 + np.sqrt(c)) ** 2
    if evals[0] <= 1.05 * edge:
        return 0
    return select_rank_gap_ratio(evals, k_max)


def spectral_sc(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    k_max: int = 4,
    sigma: float | None = None,
    c: float | None = None,
    info: dict | None = None,
) -> np.ndarray:
    """Hard-threshold spectral SC (frontier_ansatz.md Section 1 form).

    beta_j = <y1, v_j>/d_j on top-k PC scores; post predictions transport the
    donor cross-sections through the sample left basis. Rank by gap ratio;
    when sigma and c are supplied, a silence gate may force k = 0, in which
    case the forecast falls back to the constant mean(y1_pre).
    """
    T0 = y1_pre.shape[0]
    Xd = donors[:, :T0]
    U, d, Vt = np.linalg.svd(Xd, full_matrices=False)
    if sigma is not None and c is not None:
        eigs_unit = d**2 / T0
        edge = sigma**2 * (1.0 + np.sqrt(c)) ** 2
        k = 0 if eigs_unit[0] <= 1.05 * edge else select_rank_gap_ratio(eigs_unit, k_max)
    else:
        k = select_rank_gap_ratio(d**2, k_max)
    if info is not None:
        info["spectral_k"] = int(k)
    if k == 0:
        return np.full(donors.shape[1] - T0, y1_pre.mean())
    beta = (Vt[:k] @ y1_pre) / d[:k]
    sh = U[:, :k].T @ donors[:, T0:]
    return beta @ sh


def spectral_sc_full(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    sigma: float | None = None,
    c: float | None = None,
    k_max: int = 4,
    z: float = 1.959963985,
) -> dict:
    """Gated + ungated spectral SC from ONE SVD, with the preregistered 95%
    prediction interval for the gated fit (preregistration Section 2.4).

    Returns dict with pred_gated, pred_ungated, k_gated, k_ungated, ci_lo,
    ci_hi and info. When sigma/c are absent the gate is off and both
    variants coincide with the plain gap-ratio rule.
    """
    T0 = y1_pre.shape[0]
    Xd, Xp = donors[:, :T0], donors[:, T0:]
    U, d, Vt = np.linalg.svd(Xd, full_matrices=False)
    evals_unit = d**2 / T0
    k_u = select_rank_gap_ratio(evals_unit, k_max)
    k_g = k_u
    if sigma is not None and c is not None:
        edge = sigma**2 * (1.0 + np.sqrt(c)) ** 2
        k_g = 0 if evals_unit[0] <= 1.05 * edge else k_u
    Tp = Xp.shape[1]

    def _predict(k):
        if k == 0:
            ybar = float(y1_pre.mean())
            s2 = float(np.sum((y1_pre - ybar) ** 2) / T0)
            half = z * np.sqrt(s2 * (1.0 + 1.0 / T0))
            return np.full(Tp, ybar), np.full(Tp, ybar - half), np.full(Tp, ybar + half), s2
        beta = (Vt[:k] @ y1_pre) / d[:k]
        sh = U[:, :k].T @ Xp                      # k x Tp post scores
        pred = beta @ sh
        fitted = Vt[:k].T @ (beta * d[:k])        # in-sample pre fit (time space)
        rss = float(np.sum((y1_pre - fitted) ** 2))
        s2 = rss / max(T0 - k, 1)
        h = np.sum(sh**2 / d[:k, None] ** 2, axis=0)
        half = z * np.sqrt(s2 * (1.0 + h))
        return pred, pred - half, pred + half, s2

    pg, lo, hi, _ = _predict(k_g)
    pu, _, _, _ = _predict(k_u)
    return {
        "pred_gated": pg,
        "pred_ungated": pu,
        "k_gated": int(k_g),
        "k_ungated": int(k_u),
        "ci_lo": lo,
        "ci_hi": hi,
        "info": {"lambda1": float(evals_unit[0]), "edge": float(sigma**2 * (1.0 + np.sqrt(c)) ** 2) if sigma is not None and c is not None else None},
    }


def _soft_impute(M_obs, mask, lam, iters, tol=1e-4, X0=None):
    X = np.where(mask, M_obs, 0.0) if X0 is None else X0.copy()
    denom = np.linalg.norm(np.where(mask, M_obs, 0.0)) + 1e-12
    for _ in range(iters):
        U, s, Vt = np.linalg.svd(X, full_matrices=False)
        s_st = np.maximum(s - lam, 0.0)
        X_new = (U * s_st) @ Vt
        X_new[mask] = M_obs[mask]
        delta = np.linalg.norm(X_new - X) / denom
        X = X_new
        if delta < tol:
            break
    return X


def mc_nn_cv(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    lambdas=DEFAULT_MC_LAMBDAS,
    cv_holdout: float = 0.1,
    cv_seed: int = 4321,
    iters: int = 60,
    info: dict | None = None,
) -> np.ndarray:
    """Nuclear-norm MC (soft-impute) with lambda by held-out CV.

    Observed panel stacks the treated pre-row over the donors; only the
    treated post block is missing. CV masks a random fraction of observed
    entries, selects lambda by held-out MSE (warm-started down the grid),
    refits on all observed entries, reads out the completed treated-post block
    (MC-NNM-style block completion, Athey et al. 2021 setting without unit or
    time fixed-effects terms).
    """
    T0 = y1_pre.shape[0]
    T = donors.shape[1]
    M = np.vstack([np.concatenate([y1_pre, np.full(T - T0, np.nan)]), donors])
    obs = ~np.isnan(M)
    rng = np.random.default_rng(cv_seed)
    cand = np.argwhere(obs)
    n_hold = max(1, int(cv_holdout * len(cand)))
    hold_idx = cand[rng.choice(len(cand), size=n_hold, replace=False)]
    cv_mask = obs.copy()
    cv_mask[hold_idx[:, 0], hold_idx[:, 1]] = False
    M0 = np.where(obs, M, 0.0)
    scale = float(np.std(M[obs]))
    best_lam, best_err = None, np.inf
    Xw = None
    for lam in sorted(lambdas, reverse=True):
        Xi = _soft_impute(M0, cv_mask, lam * scale, iters, X0=Xw)
        Xw = Xi
        err = float(
            np.sum((Xi[hold_idx[:, 0], hold_idx[:, 1]] - M[hold_idx[:, 0], hold_idx[:, 1]]) ** 2)
        )
        if err < best_err:
            best_err, best_lam = err, lam
    Xhat = _soft_impute(M0, obs, best_lam * scale, iters)
    if info is not None:
        info["mc_lambda"] = float(best_lam)
    return Xhat[0, T0:]


def _simplex_intercept_ridge(Z, target, row_w, reg):
    """min over p in simplex(|coef|) and intercept a of
    sum_i row_w_i * (Z[i] @ coef + a - target[i])^2 + reg * ||coef||^2."""
    def obj(p):
        coef, a = p[:-1], p[-1]
        r = Z @ coef + a - target
        return float(row_w @ (r * r)) + reg * float(coef @ coef)

    def jac(p):
        coef, a = p[:-1], p[-1]
        r = Z @ coef + a - target
        gw = 2.0 * (Z.T @ (row_w * r)) + 2.0 * reg * coef
        ga = np.array([2.0 * float(row_w @ r)])
        return np.concatenate([gw, ga])

    m = Z.shape[1]
    p0 = np.r_[np.full(m, 1.0 / m), float(target.mean())]
    res = minimize(
        obj, p0, jac=jac, method="SLSQP",
        bounds=[(0.0, 1.0)] * m + [(None, None)],
        constraints=[
            {"type": "eq", "fun": lambda p: p[:-1].sum() - 1.0,
             "jac": lambda p: np.r_[np.ones(m), 0.0]}
        ],
        options={"maxiter": 600, "ftol": 1e-12},
    )
    ok = bool(res.success) or (
        res.status == 8 and abs(res.x[:-1].sum() - 1.0) < 1e-6
    )
    return res.x[:-1], res.x[-1], ok


def sdid(
    donors: np.ndarray,
    y1_pre: np.ndarray,
    reg_scale: float = 0.25,
    sweeps: int = 1,
    info: dict | None = None,
) -> np.ndarray:
    """    Reduced SDID port (Arkhangelsky et al. 2021, Algorithm 3 shape).

    Step 1 fits time weights beta (simplex + intercept) aligning pre-period
    donor levels with each donor's post average; step 2 fits unit weights
    (simplex + intercept) on beta^2-weighted pre periods; predictions are
    intercept + weighted donor outcomes. Weights enter through ROW WEIGHTS on
    residuals (never through design scaling, which would silently rescale the
    ridge penalty relative to the data term). Ridge regularization uses a
    variance heuristic (reg_scale); two alternating sweeps; no inference.
    """
    T0 = y1_pre.shape[0]
    Ypre = donors[:, :T0]
    Ypost = donors[:, T0:]
    n_d = Ypre.shape[0]
    var_y = float(np.var(np.diff(Ypre, axis=1))) + 1e-8

    beta = np.full(T0, 1.0 / T0)
    ok_all = True
    w = np.full(n_d, 1.0 / n_d)
    a = float(y1_pre.mean() - w @ Ypre.mean(axis=1))
    for _ in range(sweeps):
        beta_t, b_t, ok1 = _simplex_intercept_ridge(
            Ypre, Ypost.mean(axis=1), np.ones(n_d), reg_scale * var_y / n_d
        )
        beta = np.clip(beta_t, 1e-9, None)
        beta /= beta.sum()
        ok_all &= ok1
        w_t, a_t, ok2 = _simplex_intercept_ridge(
            Ypre.T, y1_pre, beta**2, reg_scale * var_y * float(beta @ beta)
        )
        w, a = np.clip(w_t, 0.0, None), float(a_t)
        ok_all &= ok2
    if info is not None:
        info["sdid_solver_ok"] = bool(ok_all)
    return w @ Ypost + a


# ===== flattened from code/scm_frontier/diagnostics.py =====
"""Diagnostic suite for the Idea 5 project (WP-B2).

Spectral diagnostics use pre-period data only (leakage rule). Calibration
conventions follow model_card.md: unit-space scatter, sigma^2 units,
c = n / T0.
"""

from __future__ import annotations

import numpy as np
from scipy import stats



def scree(donors_pre: np.ndarray) -> np.ndarray:
    """Descending eigenvalues of the unit-space scatter."""
    return np.sort(np.linalg.eigvalsh(unit_scatter(donors_pre)))[::-1]


def tw_mu_nu(rows: int, cols: int) -> tuple[float, float]:
    """Johnstone (2001) centering/scaling for the largest eigenvalue of a
    rows x cols iid-Gaussian matrix on the (1/cols)-scaled convention."""
    mu = (np.sqrt(rows - 1.0) + np.sqrt(cols)) ** 2 / cols
    nu = (
        (np.sqrt(rows - 1.0) + np.sqrt(cols))
        / cols
        * (1.0 / np.sqrt(rows - 1.0) + 1.0 / np.sqrt(cols)) ** (1.0 / 3.0)
    )
    return float(mu), float(nu)


def tw_statistic(donors_pre: np.ndarray, sigma: float) -> float:
    """Standardized largest eigenvalue (Tracy-Widom scaling, Johnstone 2001).

    For an n_d x T0 iid-Gaussian matrix, (lambda1 - mu)/(nu) converges to TW1,
    with mu/nu computed on the (1/T0)-scaled convention used here.
    """
    n_d, T0 = donors_pre.shape
    evals = scree(donors_pre)
    mu, nu = tw_mu_nu(n_d, T0)
    return float((evals[0] / sigma**2 - mu) / nu)


def gated_rank(evals_desc: np.ndarray, sigma: float, c: float, k_max: int = 4) -> int:
    """Silence gate (top eigenvalue <= 1.05 x MP edge => 0) plus gap ratio."""
    edge = sigma**2 * (1.0 + np.sqrt(c)) ** 2
    if evals_desc[0] <= 1.05 * edge:
        return 0
    return select_rank_gap_ratio(evals_desc, k_max)


def cv_rank_selector(
    donors_pre: np.ndarray,
    y1_pre: np.ndarray,
    k_max: int = 4,
    folds: int = 4,
    cv_seed: int = 1234,
) -> int:
    """CV-rank incumbent comparator (preregistration Section 5.2).

    Chooses k in 0..k_max minimizing contiguous-block CV MSE of the k-PC
    regression of y1_pre on donor scores (same fold stream as ridge_cv_seed).
    """
    T0 = len(y1_pre)
    U, d, Vt = np.linalg.svd(donors_pre, full_matrices=False)
    km = int(min(k_max, len(d)))
    Z_all = Vt * d[:, None]  # k x T0 score series
    rng = np.random.default_rng(cv_seed)
    perm = rng.permutation(T0)
    blocks = np.array_split(perm, folds)
    best_k, best_err = 0, np.inf
    for k in range(0, km + 1):
        err = 0.0
        Z = Z_all[:k]  # k x T0 score series
        for b in blocks:
            tr = np.setdiff1d(perm, b)
            if k == 0:
                mu_tr = y1_pre[tr].mean()
                pred = np.full(len(b), mu_tr)
            else:
                Xt = Z[:, tr]
                beta, *_ = np.linalg.lstsq(Xt.T, y1_pre[tr], rcond=None)
                pred = Z[:, b].T @ beta
            err += float(np.sum((pred - y1_pre[b]) ** 2))
        if err < best_err - 1e-12:
            best_err, best_k = err, k
    return int(best_k)


def classical_trend_ttest(y1_pre: np.ndarray) -> float:
    """Classical incumbent: two-sided OLS t-test of the linear-trend
    coefficient of the treated pre-period row (nominal level reference)."""
    T0 = len(y1_pre)
    t = np.arange(T0, dtype=float)
    X = np.column_stack([np.ones(T0), t])
    beta, *_ = np.linalg.lstsq(X, y1_pre, rcond=None)
    resid = y1_pre - X @ beta
    dof = T0 - 2
    s2 = float(resid @ resid) / dof
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
    return float(2.0 * stats.t.sf(abs(beta[1] / se), dof))


def resid_statistic(
    basis_window: np.ndarray,
    post_window: np.ndarray,
    sigma: float,
    c_basis: float,
    k_max: int = 4,
) -> tuple[float, int]:
    """Post-residual TW statistic Z (preregistration Section 8.8).

    Projects the post window off the gated spike basis of the basis window;
    returns the standardized top eigenvalue of the residual unit-space
    scatter, (lam1/sigma^2 - mu)/nu with mu/nu at aspect n_d/Tp, plus the
    gated rank used. Large Z = factor-law instability evidence.
    """
    n_d, Tb = basis_window.shape
    Tp = post_window.shape[1]
    U, d, _ = np.linalg.svd(basis_window, full_matrices=False)
    evals = d**2 / Tb
    k = gated_rank(evals, sigma, c_basis, k_max)
    if k > 0:
        R = post_window - U[:, :k] @ (U[:, :k].T @ post_window)
    else:
        R = post_window
    lam = float(np.linalg.eigvalsh((R @ R.T) / Tp)[-1]) / sigma**2
    mu, nu = tw_mu_nu(n_d, Tp)
    return float((lam - mu) / nu), int(k)


def simulated_null_z(
    n_d: int,
    Tb: int,
    Tp: int,
    sigma: float,
    c_basis: float,
    k_max: int = 4,
    G: int = 300,
    seed: int = 7_770_001,
) -> np.ndarray:
    """Sorted simulated finite-n null of the Z statistic (iid Gaussian),
    reproducible given (shapes, seed); cache ONE draw set per cell config."""
    rng = np.random.default_rng(seed)
    zs = np.empty(G)
    for g in range(G):
        B = rng.normal(0.0, sigma, size=(n_d, Tb))
        P = rng.normal(0.0, sigma, size=(n_d, Tp))
        zs[g], _ = resid_statistic(B, P, sigma, c_basis, k_max)
    return np.sort(zs)


def z_tw_pvalue(z_obs: float, null_z_sorted: np.ndarray) -> float:
    """Parametric-simulation p-value: P(null Z >= observed), +1 corrected."""
    G = len(null_z_sorted)
    exceed = int(np.searchsorted(null_z_sorted, z_obs, side="left"))
    return float((G - exceed + 1.0) / (G + 1.0))


def z_boot_pvalue(
    donors_pre: np.ndarray,
    sigma: float,
    T_post: int,
    k_max: int = 4,
    B: int = 200,
    block: int = 10,
    seed: int = 8_880_001,
) -> tuple[float, float, int]:
    """Circular block-bootstrap pre-trends test (preregistration Section 8.8).

    Observed statistic splits the pre window into a basis part (first T0 -
    Tp_eff columns) and a pseudo-post part (last Tp_eff), Tp_eff =
    min(T_post, T0 // 2). Null: B circular block resamples (length `block`)
    of the pre time index, same split. Returns (p_value, z_obs, k_used).
    """
    Y = donors_pre
    n_d, T0 = Y.shape
    Tp = int(min(T_post, T0 // 2))
    Tb = T0 - Tp
    z_obs, k_used = resid_statistic(Y[:, :Tb], Y[:, Tb:], sigma, n_d / Tb, k_max)
    rng = np.random.default_rng(seed)
    ge = 0
    for _ in range(B):
        starts = rng.integers(0, T0, size=int(np.ceil(T0 / block)))
        idx = np.concatenate([(np.arange(s, s + block) % T0) for s in starts])[:T0]
        Ystar = Y[:, idx]
        z_star, _ = resid_statistic(Ystar[:, :Tb], Ystar[:, Tb:], sigma, n_d / Tb, k_max)
        if z_star >= z_obs:
            ge += 1
    return float((ge + 1.0) / (B + 1.0)), float(z_obs), int(k_used)


def invert_bbp(lam: np.ndarray, c: float):
    """Map outlier location(s) back to spike strengths via the BBP/BGN law.

    Solves lambda = 1 + s + c + c/s for s; values at or below the bulk edge
    map to nan. Vectorized over lam.
    """
    lam = np.asarray(lam, dtype=float)
    edge = (1.0 + np.sqrt(c)) ** 2
    b = lam - 1.0 - c
    disc = b**2 - 4.0 * c
    s = np.where(disc > 0.0, (b + np.sqrt(np.maximum(disc, 0.0))) / 2.0, np.nan)
    return np.where(lam > edge, s, np.nan)


def alignment_energy(donors_pre: np.ndarray, y1_pre: np.ndarray, k: int) -> float:
    """Fraction of the treated row's pre-period energy captured by the top-k
    sample subspace, in excess of the pure-noise expectation sigma^2-normalized.

    Returns proj^2 energy share in [0, 1]: sum_{j<=k} <y1, v_j>^2 / ||y1||^2.
    """
    _, d, Vt = np.linalg.svd(donors_pre, full_matrices=False)
    kk = min(k, len(d))
    proj = Vt[:kk] @ y1_pre
    return float(proj @ proj / (y1_pre @ y1_pre))


def spike_estimates(donors_pre: np.ndarray, sigma: float, c: float, k_max: int = 4) -> dict:
    """Estimate supercritical spike strengths and the m = s/sqrt(c) ratios."""
    evals = scree(donors_pre)[: max(1, k_max)]
    s_hat = invert_bbp(evals, c)
    m_hat = s_hat / np.sqrt(c)
    return {"eigs": evals, "s_hat": s_hat, "m_hat": m_hat}


# ===== flattened from code/scm_frontier/experiment.py =====
"""Phase C cell runner (shared by all Colab notebooks and local validation).

Row contract (results_schema.yaml): one row per (rep, method) plus one
"_diag" row per rep carrying the diagnostic battery. All estimators see the
same generated panel within a replication; seeds follow preregistration
Section 3 (seed = 10000 + rep index).

Diagnostic levels (preregistration Sections 4-5):
  "none"  : estimators only.
  "light" : + gated/ungated rank, CV-rank comparator, trend t-test, scree.
  "full"  : light + Z_boot (block bootstrap) and Z_tw (simulated null,
            cached once per cell config).
"""

from __future__ import annotations

import time

import numpy as np


METHODS = (
    "donor_mean",
    "scm_simplex",
    "ridge_sc",
    "spectral_gated",
    "spectral_ungated",
    "mc_nn_cv",
    "sdid",
)

META_COLS = (
    "experiment", "shard", "cell_id", "c", "n", "T0", "T_post", "r", "m",
    "arm", "theta", "delta", "noise", "rho", "het_ratio", "rep_seed",
)
ROW_COLS = META_COLS + (
    "method", "rmse", "att_bias", "k_selected", "ridge_lambda", "mc_lambda",
    "solver_ok", "ci_cover", "z_boot_p", "z_tw_p", "trend_p", "cv_rank",
    "lambda1", "edge", "wall_ms",
)


def _meta(cell: dict, seed: int) -> dict:
    return {
        "experiment": cell["experiment"],
        "shard": cell.get("shard", ""),
        "cell_id": cell["cell_id"],
        "c": cell["c"],
        "n": cell["n"],
        "T0": cell["T0"],
        "T_post": cell["T_post"],
        "r": cell["r"],
        "m": cell["m"],
        "arm": cell["arm"],
        "theta": cell["theta"],
        "delta": cell.get("delta") if cell.get("delta") is not None else "",
        "noise": cell.get("noise", "gaussian"),
        "rho": cell.get("rho", 0.5),
        "het_ratio": cell.get("het_ratio", 4.0),
        "rep_seed": seed,
    }


def panel_kwargs(cell: dict) -> dict:
    arm = cell["arm"]
    if arm == "orthogonal":
        alignment = "none"
        share = tuple([0.0] * cell["r"])
    elif arm == "spread":
        alignment = "all"
        share = tuple([cell["theta"] / cell["r"]] * cell["r"])
    else:  # "partial" / "full" -> concentrated on factor 0
        alignment = "first"
        share = tuple([cell["theta"]] * cell["r"])
    if "spike_strengths" in cell:
        strengths = tuple(cell["spike_strengths"])
    else:
        s = cell["m"] * np.sqrt(cell["c"])
        strengths = tuple([float(s)] * cell["r"])
    return dict(
        n=cell["n"], T0=cell["T0"], T_post=cell["T_post"], r=cell["r"],
        spike_strengths=strengths,
        treated_share=share,
        alignment=alignment, sigma=1.0,
        noise=cell.get("noise", "gaussian"),
        rho=cell.get("rho", 0.5),
        het_ratio=cell.get("het_ratio", 4.0),
        structural_break=cell.get("delta"),
    )


def _empty_diag() -> dict:
    return {"z_boot_p": "", "z_tw_p": "", "trend_p": "", "cv_rank": "",
            "lambda1": "", "edge": ""}


def run_rep(cell: dict, seed: int, methods: tuple[str, ...],
            diag_level: str = "none", null_z: np.ndarray | None = None) -> list[dict]:
    """One replication -> schema rows.

    `null_z` is the per-cell cached simulated null for Z_tw; it is required
    when diag_level == "full" (compute via simulated_null_z once per cell).
    """
    pan = generate_panel(seed=seed, **panel_kwargs(cell))
    Y = pan["Y"]
    T0, T_post = cell["T0"], cell["T_post"]
    donors, y1_pre, y_star = Y[1:], Y[0, :T0], Y[0, T0:]
    c = float(cell["c"])
    sigma = 1.0
    meta = _meta(cell, seed)
    att_true = float(y_star.mean())
    rows: list[dict] = []

    need_spectral = any(m.startswith("spectral") for m in methods) or diag_level != "none"
    spec = spectral_sc_full(donors, y1_pre, sigma=sigma, c=c) if need_spectral else None

    for mname in methods:
        t0 = time.perf_counter()
        rl: object = ""
        ml: object = ""
        ok: object = ""
        k_sel: object = ""
        cover: object = ""
        if mname == "spectral_gated":
            pred, lo, hi = spec["pred_gated"], spec["ci_lo"], spec["ci_hi"]
            k_sel = spec["k_gated"]
            cover = float(np.mean((y_star >= lo) & (y_star <= hi)))
        elif mname == "spectral_ungated":
            pred = spec["pred_ungated"]
            k_sel = spec["k_ungated"]
        else:
            fn = {"donor_mean": donor_mean, "scm_simplex": scm_simplex,
                  "ridge_sc": ridge_sc, "mc_nn_cv": mc_nn_cv, "sdid": sdid}[mname]
            info: dict = {}
            pred = fn(donors, y1_pre, info=info)
            rl = info.get("ridge_lambda", "")
            ml = info.get("mc_lambda", "")
            ok = info.get("scm_success", info.get("sdid_solver_ok", ""))
        wall = (time.perf_counter() - t0) * 1000.0
        rows.append({**meta, **_empty_diag(), "method": mname,
                     "rmse": round(float(np.sqrt(np.mean((pred - y_star) ** 2))), 6),
                     "att_bias": round(float(np.mean(pred) - att_true), 6),
                     "k_selected": k_sel, "ridge_lambda": rl, "mc_lambda": ml,
                     "solver_ok": ok, "ci_cover": cover, "wall_ms": round(wall, 3)})

    if diag_level != "none":
        t0 = time.perf_counter()
        dg = _empty_diag()
        dg["cv_rank"] = cv_rank_selector(donors[:, :T0], y1_pre)
        dg["trend_p"] = round(classical_trend_ttest(y1_pre), 6)
        sc = spike_estimates(donors[:, :T0], sigma, c)
        dg["lambda1"] = round(float(sc["eigs"][0]), 6)
        dg["edge"] = round(float((1 + np.sqrt(c)) ** 2), 6)
        dg["k_selected"] = spec["k_gated"] if spec is not None else ""
        if diag_level == "full":
            pb, zb, _ = z_boot_pvalue(donors[:, :T0], sigma, T_post)
            dg["z_boot_p"] = round(pb, 6)
            dg["z_tw_p"] = round(z_tw_pvalue(zb, null_z), 6)
        rows.append({**meta, **dg, "method": "_diag", "rmse": "",
                     "att_bias": "", "ridge_lambda": "", "mc_lambda": "",
                     "solver_ok": "", "ci_cover": "",
                     "wall_ms": round((time.perf_counter() - t0) * 1000.0, 3)})
    return rows


def cell_null_z(cell: dict, G: int = 300, seed: int = 7_770_001) -> np.ndarray:
    """Simulated iid null of Z for this cell's geometry (computed once)."""
    n_d = cell["n"] - 1
    Tp_eff = int(min(cell["T_post"], cell["T0"] // 2))
    Tb = cell["T0"] - Tp_eff
    return simulated_null_z(n_d, Tb, Tp_eff, 1.0, n_d / Tb, G=G, seed=seed)


def run_cell(cell: dict, methods: tuple[str, ...], reps: int, seed_base: int,
             on_chunk, diag_level: str = "none") -> None:
    """Run a full cell, flushing rows to `on_chunk(rows)` every 25 reps."""
    null_z = cell_null_z(cell) if diag_level == "full" else None
    done = 0
    while done < reps:
        take = min(25, reps - done)
        chunk: list[dict] = []
        for j in range(done, done + take):
            chunk.extend(run_rep(cell, seed_base + j, methods, diag_level, null_z))
        on_chunk(chunk)
        done += take


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import hashlib
import json
import platform
import time
from pathlib import Path

import numpy as np
import scipy

print("python", platform.python_version(), "| numpy", np.__version__,
      "| scipy", scipy.__version__)


In [ ]:
import scm_frontier_flat as sf
NB_NAME = "nb_c1_shard36_of40"
FAMILY = "c1"
SHARD_ID = "36"
REPS = 500
SEED_BASE = 10000
CELLS = json.loads(r'''
[
 {
  "experiment": "c2i",
  "cell_id": "null_c4.0",
  "c": 4.0,
  "n": 320,
  "T0": 80,
  "T_post": 40,
  "r": 0,
  "m": 0.0,
  "arm": "orthogonal",
  "theta": 0.0,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "full",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c4.0_m0.20_partial_r3",
  "c": 4.0,
  "n": 320,
  "T0": 80,
  "T_post": 40,
  "r": 3,
  "m": 0.2,
  "arm": "partial",
  "theta": 0.25,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c4.0_m1.20_orthogonal_r1",
  "c": 4.0,
  "n": 320,
  "T0": 80,
  "T_post": 40,
  "r": 1,
  "m": 1.2,
  "arm": "orthogonal",
  "theta": 0.0,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c4.0_m2.40_partial_r3",
  "c": 4.0,
  "n": 320,
  "T0": 80,
  "T_post": 40,
  "r": 3,
  "m": 2.4,
  "arm": "partial",
  "theta": 0.25,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c0.5_m0.80_partial_r3",
  "c": 0.5,
  "n": 113,
  "T0": 226,
  "T_post": 113,
  "r": 3,
  "m": 0.8,
  "arm": "partial",
  "theta": 0.25,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c0.5_m2.00_partial_r1",
  "c": 0.5,
  "n": 113,
  "T0": 226,
  "T_post": 113,
  "r": 1,
  "m": 2.0,
  "arm": "partial",
  "theta": 0.25,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c2.0_m0.60_orthogonal_r3",
  "c": 2.0,
  "n": 226,
  "T0": 113,
  "T_post": 56,
  "r": 3,
  "m": 0.6,
  "arm": "orthogonal",
  "theta": 0.0,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 },
 {
  "experiment": "c1",
  "cell_id": "c2.0_m1.80_orthogonal_r1",
  "c": 2.0,
  "n": 226,
  "T0": 113,
  "T_post": 56,
  "r": 1,
  "m": 1.8,
  "arm": "orthogonal",
  "theta": 0.0,
  "delta": null,
  "methods": [
   "donor_mean",
   "scm_simplex",
   "ridge_sc",
   "spectral_gated",
   "spectral_ungated",
   "mc_nn_cv"
  ],
  "diag": "light",
  "reps": 500
 }
]
''')
OUT = Path('idea5_out'); OUT.mkdir(exist_ok=True)
T_NB = time.perf_counter()


def make_flush(csv_path, cols):
    import csv as _csv
    def flush(rows):
        new = not Path(csv_path).exists()
        with open(csv_path, "a", newline="") as fh:
            w = _csv.DictWriter(fh, fieldnames=cols)
            if new:
                w.writeheader()
            w.writerows(rows)
    return flush


In [ ]:
CSV = OUT / f"{NB_NAME}.csv"
flush = make_flush(CSV, sf.ROW_COLS)

META = {"notebook": NB_NAME, "family": FAMILY, "shard": SHARD_ID,
        "cells": [{"cell_id": c["cell_id"], "experiment": c["experiment"]}
                  for c in CELLS],
        "reps_per_cell": REPS, "seed_base": SEED_BASE,
        "versions": {"python": platform.python_version(),
                     "numpy": np.__version__},
        "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "cell_errors": {}, "rows_expected": None}

META["rows_expected"] = sum(
    c["reps"] * (len(c["methods"]) + (1 if c["diag"] != "none" else 0))
    for c in CELLS)
print(f"{NB_NAME}: {len(CELLS)} cells, {META['rows_expected']} expected rows")

for ci, cell in enumerate(CELLS):
    t0 = time.perf_counter()
    try:
        sf.run_cell(cell, tuple(cell["methods"]), cell["reps"], SEED_BASE,
                    on_chunk=flush, diag_level=cell["diag"])
    except Exception as exc:  # record, continue with remaining cells
        META["cell_errors"][cell["cell_id"]] = repr(exc)
        print(f"[{ci + 1}/{len(CELLS)}] {cell['cell_id']} ERROR {exc!r}")
        continue
    print(f"[{ci + 1}/{len(CELLS)}] {cell['cell_id']} done in "
          f"{time.perf_counter() - t0:.1f}s")
print(f"all cells done in {time.perf_counter() - T_NB:.0f}s")

In [ ]:
import gzip
import shutil

rows_written = sum(1 for _ in open(CSV)) - 1
META["rows_written"] = rows_written
META["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
META["wall_s"] = round(time.perf_counter() - T_NB, 1)
if META["cell_errors"]:
    print("WARNING: cells with errors ->", META["cell_errors"])
assert rows_written == META["rows_expected"] or META["cell_errors"], (
    f"incomplete without recorded errors: {rows_written} "
    f"!= {META['rows_expected']}")
META["csv_sha256"] = hashlib.sha256(open(CSV, "rb").read()).hexdigest()
gz = CSV.with_suffix(".csv.gz")
with open(CSV, "rb") as fin, gzip.open(gz, "wb") as fout:
    shutil.copyfileobj(fin, fout)
meta_path = OUT / f"{NB_NAME}_meta.json"
meta_path.write_text(json.dumps(META, indent=1))
print("rows:", rows_written, "| sha256:", META["csv_sha256"][:16], "...")

try:
    from google.colab import files
    files.download(str(gz))
    files.download(str(meta_path))
    print("Downloaded:", gz.name, meta_path.name)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
